<a href="https://colab.research.google.com/github/ozodbekAI/Data-Scince-and-AI-Portfolio/blob/main/RNN_HI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim

In [ ]:
sequence = 'salom'
chars = sorted(list(set(sequence)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

In [ ]:
x_data = [char_to_idx[ch] for ch in sequence[:-1]]
y_data = [char_to_idx[ch] for ch in sequence[1:]]

In [ ]:
x = torch.tensor(x_data).unsqueeze(1)
y = torch.tensor(y_data)

In [ ]:
class CharsRNN(nn.Module):
  def __init__(self, vocab_size, hidden_size):
    super(CharsRNN, self).__init__()
    self.rnn = nn.RNN(vocab_size, hidden_size, batch_first=True)
    self.fc = nn.Linear(hidden_size, vocab_size)

  def forward(self, x, hidden):
    out, hidden = self.rnn(x, hidden)
    out = self.fc(out.squeeze(1))
    return out, hidden

In [ ]:
vocab_size = len(chars)
hidden_size = 8
model = CharsRNN(vocab_size, hidden_size)

In [ ]:
def one_hot(index, vocab_size):
  vec = torch.zeros(1, 1, vocab_size)
  vec[0, 0, index] = 1
  return vec

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
for epoch in range(100):
   loss = 0
   h = torch.zeros(1, 1, hidden_size)
   for i in range(len(x)):
     input_vec = one_hot(x[i], vocab_size)
     out, h = model(input_vec, h)
     loss += criterion(out, y[i].unsqueeze(0))
   optimizer.zero_grad()
   loss.backward()
   optimizer.step()

   if epoch % 10 == 0:
     pred_seq = ''
     h_test = torch.zeros(1, 1, hidden_size)
     for i in range(len(x)):
       input_vec = one_hot(x[i], vocab_size)
       out, h_test = model(input_vec, h_test)
       pred_idx = torch.argmax(out, dim=1).item()
       pred_char = idx_to_char[pred_idx]
       pred_seq += pred_char
     print(f'Epoch: {epoch}, Loss: {loss.item()}, Pred: {pred_seq}')